In [52]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

#Determining parameters, same as ellips simulation
parameters : {
'r_m' : 0.0064,       # Motor pulley radius [m]
'L_m' : 0.003,      # Motor inductance [H]
'R_m' : 1.8,        # Motor resistance [Ohm]
'J_m' : 5.7e-6,      # Motor rotor inertia [kg*m^2]
'K_m' : 0.22,       # Motor constant [Nm/A]
'm' : 0.6,          # Tool head mass [kg]
'J_tot' : 18e-6,  # Total effective inertia of the system
'Max_x' : 0.22,       # Maximum x position [m]
'Max_y' : 0.22,       # Maximum y position [m]
'mu_r' : 0.003,     # Bearing rotational friction [Nm*s/rad]
'mu_lx' : 3,      # Linear friction x-axis [N*s/m]
'mu_ly' : 3,      # Linear friction y-axis [N*s/m]

'C_tn': 15.0,         # Thermische capaciteit nozzle [J/K]
'R_tn': 12.0,        # Thermische weerstand [K/W]
'C_n': 100e-6,       # Condensator nozzle [F]
'R_n': 5.0,          # Weerstand nozzle [Ohm]
'L_n': 0.01,         # Inductie nozzle [H]
'T_c': 20.0,        #Temperatuur omgeving [Celcius]

'T_high': 200.0,     # Gewenste printtemperatuur [°C]
'T_low': 40.0,       # Veilige wissel/afkoeltemperatuur [°C]
'x_wipe': 0.0,       # Wipe tower x-positie [m]
'y_wipe': 0.0,       # Wipe tower y-positie [m]
}


#Gemini computed the matrices from a picture from our report
def get_state_space_matrices(p):

    A = np.zeros((9, 9))
    
    A[0, 1] = 1.0
    A[1, 1] = -(p['mu_r'] / p['J_tot'] + (p['mu_lx'] * p['r_m']**2) / (2 * p['J_tot']))
    A[1, 2] = (p['K_m'] * p['r_m']) / p['J_tot']
    A[2, 1] = -p['K_m'] / (p['L_m'] * p['r_m'])
    A[2, 2] = -p['R_m'] / p['L_m']
    
    A[3, 4] = 1.0
    A[4, 4] = -(p['mu_r'] / p['J_tot'] + (p['mu_ly'] * p['r_m']**2) / (2 * p['J_tot']))
    A[4, 5] = (p['K_m'] * p['r_m']) / p['J_tot']
    A[5, 4] = -p['K_m'] / (p['L_m'] * p['r_m'])
    A[5, 5] = -p['R_m'] / p['L_m']
    
    A[6, 6] = -1.0 / (p['C_tn'] * p['R_tn'])
    A[6, 7] = 1.0 / (p['C_tn'] * p['R_n'])
    A[7, 7] = -1.0 / (p['C_n'] * p['R_n'])
    A[7, 8] = 1.0 / p['C_n']
    A[8, 7] = -1.0 / p['L_n']
    
    B = np.zeros((9, 3))
    B[2, 0] = 1.0 / p['L_m']
    B[5, 1] = 1.0 / p['L_m']
    B[8, 2] = 1.0 / p['L_n']
    
    E = np.zeros((9, 1))
    E[6, 0] = p['T_c'] / (p['C_tn'] * p['R_tn'])
    
    M = np.array([
        [0.5,  0.5],
        [0.5, -0.5]
    ])
    
    return A, B, E, M

def state_space_system(t, current_fsm_state, u_input, p):
    A,B,E,M = get_state_space_matrices(p)

    x = state.reshape((9, 1))
    u = np.zeros((3,1))

    if current_fsm_state == 'PRINT':
        u_x, u_y, u_n = 3.0, 3.0, 12.0
        
    elif current_fsm_state == 'MOVE_TO_WIPE':
        u_x = -5.0 * state[0]  # P-regeling naar x=0
        u_y = -5.0 * state[3]  # P-regeling naar y=0
        u_n = 12.0 
    
    elif current_fsm_state == 'WIPE':
        u_x, u_y, u_n = 0.0, 0.0, 12.0
        
    elif current_fsm_state == 'HEAT_OFF':
        u_x, u_y, u_n = 0.0, 0.0, 0.0
        
    elif current_fsm_state == 'LOAD':
        u_x, u_y, u_n = 0.0, 0.0, 0.0
        
    elif current_fsm_state == 'HEAT_ON':
        u_x, u_y, u_n = 0.0, 0.0, 12.0

    else:
        u_x, u_y, u_n = 0.0, 0.0, 0.0

    u = np.array([[u_x], [u_y], [u_n]])

    dxdt = A @ x + B @ u + E
    return dxdt.flatten()

fsm_sequence = ['PRINT', 'MOVE_TO_WIPE', 'WIPE', 'HEAT_OFF', 'LOAD', 'HEAT_ON']
durations = [0.5, 0.4, 0.2, 1.0, 0.5, 1.0] 

x_curr = np.zeros(9)
x_curr[6] = parameters['T_c']
t_start = 0.0

t_all = []
y_all = []

for current_fsm_state in fsm_sequence:
    t_span = (t_start, t_start + durations[fsm_sequence.index(current_fsm_state)])
    t_eval = np.linspace(t_span[0], t_span[1], 50)
    
    sol = solve_ivp(state_space_system, t_span, x_curr, args=(current_fsm_state, parameters), t_eval=t_eval)
    
    t_all.append(sol.t)
    y_all.append(sol.y)
    
    x_curr = sol.y[:, -1]
    t_start = t_span[1]

t_total = np.concatenate(t_all)
y_total = np.hstack(y_all)



fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

axes[0].plot(t_total, y_total[0], label='$x$ [m]', color='tab:blue', linewidth=2)
axes[0].plot(t_total, y_total[3], label='$y$ [m]', color='tab:orange', linewidth=2)
axes[0].set_ylabel('Position [m]')
axes[0].grid(True)
axes[0].legend(loc='upper left')

axes[1].plot(t_total, y_total[2], label='$i_x$ [A]', color='tab:green', linewidth=2)
axes[1].plot(t_total, y_total[5], label='$i_y$ [A]', color='tab:red', linewidth=2)
axes[1].set_ylabel('Current [A]')
axes[1].grid(True)
axes[1].legend(loc='upper left')

axes[2].plot(t_total, y_total[6], label='$T_n$ [°C]', color='tab:purple', linewidth=2)
axes[2].plot(t_total, y_total[7], label='$u_{C_n}$ [V]', color='tab:brown', linewidth=2)
axes[2].set_xlabel('Time [s]')
axes[2].set_ylabel('Nozzle ')
axes[2].grid(True)
axes[2].legend(loc='upper left')

plt.tight_layout()
plt.show()


NameError: name 'state' is not defined